# AIC 2026 - Search + API + Nộp bài

Chạy tuần tự từ trên xuống.

| Cell | Việc |
|---|---|
| 1–5 | search (code gốc `searcher.py`) |
| 6–8 | API + tunnel → `aic.verse.id.vn` (cho web) |
| 9–10 | xuất `aic-workspace.json` + `submission.zip` |

**Cần trước khi chạy:** attach 2 dataset (data BTC + json-npy), bật Internet, Accelerator = GPU T4 x2.

**Kaggle Secrets:** `GOOGLE_API_KEY` (cho LLM), `CF_TUNNEL_TOKEN` (cho hostname cố định).

In [ ]:
# ===== CELL 1 - cài thư viện =====
!pip install -q faiss-cpu rank-bm25 sentence-transformers google-genai pydantic

In [ ]:
# ===== CELL 2 - trỏ tới dữ liệu =====
# Dữ liệu đã là Kaggle Dataset -> KHÔNG cần snapshot_download, KHÔNG cần HF token.
import glob, os

# Đặt tay nếu muốn chắc chắn:
DATABASE_ROOT_PATH = "/kaggle/input/datasets/cbg6682/npy-json/downloaded_data"

# Không có thì tự dò: tìm một file *_caption.npy bất kỳ rồi đi ngược lên 3 cấp
# (<root>/Videos_Lxx/<video_id>/<video_id>_caption.npy)
if not os.path.isdir(DATABASE_ROOT_PATH):
    hit = None
    for depth in range(2, 9):
        for root in ("/kaggle/input", "/kaggle/working"):
            found = [f for f in glob.glob(os.path.join(root, *(["*"] * depth), "*_caption.npy"))
                     if "/.cache/" not in f]
            if found:
                hit = found[0]
                break
        if hit:
            break
    if not hit:
        raise FileNotFoundError(
            "Không thấy file *_caption.npy nào dưới /kaggle/input hay /kaggle/working.\n"
            "Đã attach dataset chứa feature BGE-M3 chưa?"
        )
    DATABASE_ROOT_PATH = os.path.dirname(os.path.dirname(os.path.dirname(hit)))

n = len(glob.glob(f"{DATABASE_ROOT_PATH}/*/*/*.json"))
print("DATABASE_ROOT_PATH =", DATABASE_ROOT_PATH)
print(f"số video: {n}   {'✅' if n else '❌ sai đường dẫn - sửa DATABASE_ROOT_PATH ở trên'}")

In [ ]:
# ===== CELL 3 — phân tích prompt bằng LLM =====
# Nguyên văn searcher.py. Khác đúng một chỗ: API key đọc từ Kaggle Secrets thay vì hardcode.
import os
import json
import pandas as pd
from typing import List, Optional
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# ---------------------------------------------------------------------------
# 1. DANH SÁCH API KEY VÀ MODEL FALLBACK
# ---------------------------------------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    API_KEYS = [UserSecretsClient().get_secret("GOOGLE_API_KEY")]
except Exception:
    API_KEYS = [
        "",
        # Add more backup keys if needed
    ]

MODELS = [
    "models/gemini-3.1-flash-lite",
    "models/gemini-3.5-flash-lite",
    "models/gemma-4-31b-it",
    "models/gemma-4-26b-a4b-it"
]


# ---------------------------------------------------------------------------
# 2. ĐỊNH NGHĨA STRUCTURAL SCHEMA & SYSTEM PROMPT
# ---------------------------------------------------------------------------
class ActionQuery(BaseModel):
    spatial_context_keywords: List[str] = Field(
        default_factory=list,
        description="Danh sách 3-6 từ khóa đơn lẻ (TIẾNG ANH) mô tả các vật thể/đối tượng chính trong cảnh. Mỗi từ khóa là 1-2 từ ngắn gọn. Ví dụ: ['dam', 'river', 'rain', 'aerial view', 'mountain']."
    )
    spatial_context_attributes: str = Field(
        default="",
        description="Câu ngắn TIẾNG ANH kiểu COCO caption (~10-15 từ) mô tả bối cảnh + các thuộc tính nổi bật (màu sắc, số lượng, vị trí). Ví dụ: 'A man in a dark blue suit holding a large gray rock near his face'."
    )
    spatial_context_detailed: str = Field(
        default="",
        description="Câu TIẾNG ANH dịch sát nghĩa, đầy đủ chi tiết hành động và bối cảnh từ prompt gốc. Đây là bản dịch trung thành nhất, bao gồm mệnh đề phụ nếu cần."
    )
    spatial_context_fallback: str = Field(
        default="",
        description="Câu TIẾNG ANH cực kỳ khái quát (~5-8 từ), chỉ mô tả loại cảnh/sự kiện ở mức category cha. Ví dụ: 'outdoor natural landscape', 'people at a gas station', 'close-up of a document'."
    )
    ocr_text: List[str] = Field(
        default_factory=list,
        description="List các đoạn chữ viết xuất hiện trên màn hình tại cảnh này. Giữ nguyên ngôn ngữ được miêu tả trong prompt, không tự dịch. Bỏ các từ gọi tên vật thể (như 'bảng', 'biển')."
    )
    asr_text: List[str] = Field(
        default_factory=list,
        description="List các nội dung lời nói, thuyết minh hoặc âm thanh (nằm sau cụm 'nói về', 'nhắc đến'). Giữ nguyên ngôn ngữ được miêu tả trong prompt, không tự dịch."
    )

class QueryAnalysisResult(BaseModel):
    reasoning_process: str = Field(
        description="Suy luận: Ghi nhận Task Type. Phân tích các sự kiện và SẮP XẾP LẠI theo ĐÚNG TRÌNH TỰ THỜI GIAN nếu người dùng kể lộn xộn. Trích xuất, dịch sang tiếng Anh cho hình ảnh."
    )
    task_type: int = Field(
        description="Loại Task: 1 (Tìm 1 hoặc nhiều cảnh tĩnh mô tả độc lập), 2 (VQA - Trả lời câu hỏi), 3 (Tìm chuỗi nhiều sự kiện/cảnh nối tiếp nhau)."
    )
    actions: List[ActionQuery] = Field(
        description="Danh sách các cảnh. Nếu mô tả nhiều cảnh khác nhau, TÁCH THÀNH NHIỀU ACTION độc lập. BẮT BUỘC sắp xếp các phần tử theo đúng trình tự thời gian xảy ra trong video từ trước đến sau."
    )
    questions: List[str] = Field(
        default_factory=list,
        description="Danh sách câu hỏi cần giải đáp (Chỉ cho Task 2). Task 1 hoặc 3 để rỗng []."
    )

# ---------------------------------------------------------------------------
# 3. SYSTEM PROMPT THIẾT KẾ CHO GEMINI (Multi-Level Spatial Context)
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """Bạn là Bộ điều phối Phân tích Truy vấn Video Đa phương thức (Multi-Modal Video Query Orchestrator).
Nhiệm vụ của bạn là nhận Prompt từ người dùng, thực hiện suy luận từng bước (Chain-of-Thought) để bóc tách thông tin và SẮP XẾP LẠI các cảnh theo tiến trình thời gian.

QUY TRÌNH SUY LUẬN TỪNG BƯỚC (Điền vào `reasoning_process`):

BƯỚC 1: XÁC ĐỊNH TASK (1, 2, hay 3)
- Nếu người dùng ĐÃ CHỈ ĐỊNH SẴN loại Task, BẠN PHẢI DÙNG GIÁ TRỊ ĐÓ.
- Nếu không: Task 3 (Chuỗi sự kiện), Task 2 (Có câu hỏi), Task 1 (Mô tả cảnh tĩnh).

BƯỚC 2: TÁCH CẢNH (CHIA ACTIONS) - QUY TẮC SỐNG CÒN
- TUYỆT ĐỐI KHÔNG GỘP NHIỀU BỐI CẢNH VÀO 1 ACTION. Bạn BẮT BUỘC phải tách thành nhiều `ActionQuery` riêng biệt nếu:
  1. Có nhiều hành động xảy ra nối tiếp nhau.
  2. Có sự thay đổi về tiêu điểm (focus) hoặc góc máy. Ví dụ: Nếu prompt mô tả một bối cảnh chung/hoạt động chung (cảnh toàn), sau đó đi sâu vào mô tả chi tiết một vật thể cầm tay, một tài liệu, hoặc một màn hình (cảnh cận/zoom) -> ĐÂY BẮT BUỘC PHẢI LÀ 2 ACTION KHÁC NHAU.
- Sắp xếp các Action theo đúng trình tự thời gian xảy ra. (Nếu kể ngược thì tự đảo lại).

BƯỚC 3: PHÂN LẬP NGÔN NGỮ & THÔNG TIN (CHO TỪNG ACTION) - MULTI-LEVEL SPATIAL CONTEXT
Cho mỗi Action, bạn PHẢI điền ĐẦY ĐỦ 4 trường mô tả hình ảnh ở 4 MỨC ĐỘ TRỪU TƯỢNG KHÁC NHAU. Tất cả đều BẰNG TIẾNG ANH.

  ► `spatial_context_keywords`: 3-6 từ khóa đơn lẻ (1-2 từ/keyword). Đây là các vật thể, khái niệm chính.
    Ví dụ: ["dam", "river", "rain", "aerial view"]

  ► `spatial_context_attributes`: 1 câu ngắn (~10-15 từ) kiểu COCO caption, tập trung vào vật thể + thuộc tính nổi bật (màu sắc, số lượng, vị trí tương đối).
    Ví dụ: "A man in a dark blue suit holding a large rough gemstone"

  ► `spatial_context_detailed`: 1 câu dịch sát nghĩa, đầy đủ nhất, giữ nguyên mọi chi tiết hành động + bối cảnh từ prompt gốc. Có thể dài, nhiều mệnh đề phụ.
    Ví dụ: "The film begins with a map showing hydraulic structures appearing four times, then cuts to an aerial shot of a dam, followed by a close-up of the dam in the rain"

  ► `spatial_context_fallback`: 1 cụm cực ngắn (~5-8 từ), chỉ nêu loại cảnh/category cha. Dùng khi các mức chi tiết hơn không khớp.
    Ví dụ: "outdoor natural landscape", "people at a gas station"

- Lời nói (`asr_text`): Âm thanh, lời nói nghe được. Giữ nguyên ngôn ngữ.
- Chữ viết (`ocr_text`): BẤT CỨ NỘI DUNG CHỮ NÀO CÓ THỂ ĐỌC ĐƯỢC BẰNG MẮT trong video. Nó không chỉ là văn bản chèn trên video, mà bao gồm cả chữ in trên các vật thể vật lý (ví dụ: thông tin ghi trên giấy tờ, sách báo, bảng biểu, công thức, bao bì). Nếu prompt trích dẫn thông tin từ các vật thể này, nội dung đó chính là OCR. Giữ nguyên ngôn ngữ.

QUY TẮC BẮT BUỘC:
- Task 1 và 3: `questions` BẮT BUỘC rỗng `[]`.
- KHÔNG đưa nội dung văn bản (OCR) hoặc lời nói (ASR) vào các trường hình ảnh `spatial_context_*`. Hình là hình, tiếng là tiếng, chữ là chữ.
- BẮT BUỘC điền cả 4 trường spatial_context cho mỗi Action (trừ khi cảnh thực sự không có thông tin hình ảnh).
"""

# ---------------------------------------------------------------------------
# 4. HÀM PHÂN TÍCH PROMPT CHÍNH ĐÃ BỔ SUNG THAM SỐ `explicit_task_type`
# ---------------------------------------------------------------------------
def analyze_prompt(prompt: str, explicit_task_type: Optional[int] = None) -> dict:
    """
    Thực hiện phân tích prompt sử dụng LLM và trả về dict chứa thông tin được yêu cầu.

    Args:
        prompt (str): Câu truy vấn gốc của người dùng.
        explicit_task_type (int, optional): Truyền 1, 2, hoặc 3 nếu đã biết trước loại Task (sẽ ép LLM dùng loại này và bỏ qua suy luận Task). Mặc định là None.
    """
    last_error = None

    final_prompt = prompt
    if explicit_task_type in [1, 2, 3]:
        final_prompt = (
            f"[SYSTEM INSTRUCTION BỔ SUNG TỪ HỆ THỐNG]: Đã ấn định truy vấn này thuộc TASK {explicit_task_type}. "
            f"Bạn hãy gán thẳng `task_type` = {explicit_task_type}, KHÔNG CẦN TỰ SUY LUẬN BƯỚC 1. "
            f"Chỉ tập trung vào bóc tách thời gian, không gian, âm thanh, chữ viết cho nội dung sau đây:\n\n"
            f"NỘI DUNG TRUY VẤN: \"{prompt}\""
        )

    for key in API_KEYS:
        if not key or key.startswith("YOUR_KEY"):
            continue

        client = genai.Client(api_key=key)

        for model_name in MODELS:
            try:
                config = types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.0,
                    top_k=1,
                    top_p=0.1,
                    response_mime_type="application/json",
                    response_schema=QueryAnalysisResult,
                )

                response = client.models.generate_content(
                    model=model_name,
                    contents=final_prompt,
                    config=config,
                )

                result_dict = json.loads(response.text)
                return result_dict

            except Exception as e:
                last_error = e
                print(f"[Warning] Thử nghiệm thất bại với Model {model_name} / Key ...{key[-4:]}: {e}")
                continue

    raise RuntimeError(f"Tất cả API Key và Model đều không thể xử lý truy vấn. Lỗi cuối cùng: {last_error}")

In [ ]:
# ===== CELL 4 — search engine (nguyên văn searcher.py) =====
import os
import json
import numpy as np
import pandas as pd
import faiss
import torch
from typing import Dict, List, Optional, Set
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi


class VideoSearchEngine:
    def __init__(self, db_path: str, bge_model_name: str = 'BAAI/bge-m3', clip_model_name: str = 'clip-ViT-B-32'):
        print("⏳ Khởi tạo Search Engine...")
        self.db_path = db_path
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        print(f"   🧠 [1/2] Đang nạp model BGE-M3 ({self.device})...", flush=True)
        self.bge_model = SentenceTransformer(bge_model_name, device=self.device)
        print(f"   👁️ [2/2] Đang nạp model CLIP ({self.device})...", flush=True)
        self.clip_model = SentenceTransformer(clip_model_name, device=self.device)
        print("   ✅ Đã nạp xong 2 model BGE-M3 & CLIP!", flush=True)

        self.metadata = []

        self.dim_bge = 1024
        self.dim_clip = 512

        self.index_caption = faiss.IndexFlatIP(self.dim_bge)
        self.index_speech = faiss.IndexFlatIP(self.dim_bge)
        self.index_ocr = faiss.IndexFlatIP(self.dim_bge)
        self.index_clip = faiss.IndexFlatIP(self.dim_clip)

        self.ocr_to_frame_map = []

        self.corpus_caption = []
        self.corpus_speech = []
        self.corpus_ocr = []

        self._load_database()
        self._build_bm25_indices()

    def _load_database(self):
        print(f"📥 Đang load dữ liệu từ Database: {self.db_path} ...")
        caption_vectors, speech_vectors, ocr_vectors, clip_vectors = [], [], [], []
        global_frame_idx = 0

        if not os.path.exists(self.db_path):
            print(f"⚠️ Không tìm thấy đường dẫn {self.db_path}")
            return

        for folder_name in os.listdir(self.db_path):
            folder_path = os.path.join(self.db_path, folder_name)
            if not os.path.isdir(folder_path):
                continue

            for video_id in os.listdir(folder_path):
                video_path = os.path.join(folder_path, video_id)
                if not os.path.isdir(video_path):
                    continue

                json_path = os.path.join(video_path, f"{video_id}.json")
                if not os.path.exists(json_path):
                    continue

                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                try:
                    clip_npy = np.load(os.path.join(video_path, f"{video_id}.npy"))
                    cap_npy = np.load(os.path.join(video_path, f"{video_id}_caption.npy"))
                    speech_npy = np.load(os.path.join(video_path, f"{video_id}_speech.npy"))
                    ocr_npy = np.load(os.path.join(video_path, f"{video_id}_ocr.npy"), allow_pickle=True)
                except Exception as e:
                    print(f"   ⚠️ Lỗi load feature của {video_id}: {e}. Bỏ qua video này.")
                    continue

                for i, frame in enumerate(data['frames']):
                    frame['video_id'] = video_id
                    frame['folder_name'] = folder_name
                    frame['global_idx'] = global_frame_idx
                    self.metadata.append(frame)

                    self.corpus_caption.append(frame.get('visual_caption', '').lower().split())
                    self.corpus_speech.append(frame.get('speech_text', '').lower().split())
                    self.corpus_ocr.append(frame.get('ocr_text', '').replace('|', ' ').lower().split())

                    clip_vectors.append(clip_npy[i])
                    caption_vectors.append(cap_npy[i])
                    speech_vectors.append(speech_npy[i])

                    frame_ocr_vecs = ocr_npy[i]
                    if frame_ocr_vecs.shape[0] > 0 and frame_ocr_vecs.shape[1] == self.dim_bge:
                        for vec in frame_ocr_vecs:
                            ocr_vectors.append(vec)
                            self.ocr_to_frame_map.append(global_frame_idx)

                    global_frame_idx += 1

        if clip_vectors:
            clip_np = np.vstack(clip_vectors).astype('float32')
            faiss.normalize_L2(clip_np)
            self.index_clip.add(clip_np)

        if caption_vectors:
            cap_np = np.vstack(caption_vectors).astype('float32')
            faiss.normalize_L2(cap_np)
            self.index_caption.add(cap_np)

        if speech_vectors:
            speech_np = np.vstack(speech_vectors).astype('float32')
            faiss.normalize_L2(speech_np)
            self.index_speech.add(speech_np)

        if ocr_vectors:
            ocr_np = np.vstack(ocr_vectors).astype('float32')
            faiss.normalize_L2(ocr_np)
            self.index_ocr.add(ocr_np)

        print(f"✅ Hoàn tất load {len(self.metadata)} frames vào hệ thống!")

    def _build_bm25_indices(self):
        print("🧠 Đang khởi tạo thuật toán BM25 (song song 3 luồng)...")
        if self.corpus_caption:
            from concurrent.futures import ThreadPoolExecutor
            with ThreadPoolExecutor(max_workers=3) as executor:
                fut_cap = executor.submit(BM25Okapi, self.corpus_caption)
                fut_sp = executor.submit(BM25Okapi, self.corpus_speech)
                fut_ocr = executor.submit(BM25Okapi, self.corpus_ocr)
                self.bm25_caption = fut_cap.result()
                self.bm25_speech = fut_sp.result()
                self.bm25_ocr = fut_ocr.result()

    def _min_max_scale(self, scores: np.ndarray) -> np.ndarray:
        if len(scores) == 0 or np.max(scores) == np.min(scores):
            return np.zeros_like(scores, dtype=float)
        return (scores - np.min(scores)) / (np.max(scores) - np.min(scores))

    def _modality_confidence(self, score_arr: np.ndarray, bg_k: int = 20) -> float:
        if score_arr.size == 0:
            return 0.0
        ordered = np.sort(score_arr)[::-1]
        top1 = ordered[0]
        bg = ordered[1:bg_k + 1] if ordered.size > 1 else ordered
        bg_mean = float(np.mean(bg)) if bg.size > 0 else 0.0
        return max(float(top1) - bg_mean, 0.0)

    def _fuse_weights(self, base_weights: Dict[str, float], active_modalities: List[str], component_scores: Dict[str, np.ndarray]) -> Dict[str, float]:
        if not active_modalities:
            return {}
        conf = {m: self._modality_confidence(component_scores[m]) for m in active_modalities}
        conf_sum = sum(conf.values())
        eff = {}
        for m in active_modalities:
            conf_norm = conf[m] / conf_sum if conf_sum > 1e-8 else 1.0 / len(active_modalities)
            eff[m] = base_weights.get(m, 1.0) * (0.5 + conf_norm)
        total = sum(eff.values())
        if total <= 1e-8:
            uniform = 1.0 / len(active_modalities)
            return {m: uniform for m in active_modalities}
        return {m: eff[m] / total for m in active_modalities}

    def _get_region_mask(self, search_regions: Optional[List[str]] = None) -> Optional[np.ndarray]:
        """Tạo boolean mask để lọc chỉ các frame thuộc search_regions (tên thư mục).
        Trả về None nếu không cần lọc (tìm toàn bộ)."""
        if not search_regions:
            return None
        region_set = set(search_regions)
        mask = np.array([meta.get('folder_name', '') in region_set for meta in self.metadata], dtype=bool)
        n_match = int(np.sum(mask))
        print(f"   🔍 Region filter: {len(region_set)} vùng → {n_match}/{len(self.metadata)} frames được giữ lại")
        return mask

    def _apply_region_mask(self, scores: np.ndarray, mask: Optional[np.ndarray]) -> np.ndarray:
        """Gán điểm -inf cho các frame ngoài vùng tìm kiếm."""
        if mask is None:
            return scores
        masked = scores.copy()
        masked[~mask] = -1e9
        return masked

    def _collect_spatial_texts(self, action: Dict) -> List[str]:
        """Thu thập tất cả spatial context texts từ các trường multi-level của 1 action.
        Hỗ trợ backward-compatible với trường `spatial_context` cũ (str hoặc List[str])."""
        texts = []

        # Trường mới: multi-level
        for field in ['spatial_context_fallback', 'spatial_context_attributes', 'spatial_context_detailed']:
            val = action.get(field, '')
            if val and isinstance(val, str) and val.strip():
                texts.append(val.strip())

        kw = action.get('spatial_context_keywords', [])
        if kw and isinstance(kw, list) and len(kw) > 0:
            texts.append(", ".join(kw))

        # Backward-compatible: trường `spatial_context` cũ
        old = action.get('spatial_context', None)
        if old:
            if isinstance(old, str) and old.strip():
                texts.append(old.strip())
            elif isinstance(old, list):
                texts.extend([t.strip() for t in old if isinstance(t, str) and t.strip()])

        return texts

    def _score_visual_ensemble(self, spatial_texts: List[str], num_frames: int, scores_dict: Dict[str, np.ndarray]):
        """Prompt-ensembling: encode từng text ở mỗi mức trừu tượng,
        rồi lấy max score per-frame (thay vì mean embedding) cho CLIP & BGE.
        BM25 ghép tất cả text lại thành 1 bag-of-words."""

        # --- CLIP: max-pool across all perspectives ---
        clip_all = np.zeros((len(spatial_texts), num_frames))
        clip_embs = self.clip_model.encode(spatial_texts, normalize_embeddings=True).astype('float32')
        for t_idx in range(len(spatial_texts)):
            emb = clip_embs[t_idx:t_idx+1]
            D, I = self.index_clip.search(emb, num_frames)
            for dist, idx in zip(D[0], I[0]):
                clip_all[t_idx, idx] = dist
        scores_dict['faiss_clip'] = np.max(clip_all, axis=0)

        # --- BGE (Caption): max-pool across all perspectives ---
        bge_all = np.zeros((len(spatial_texts), num_frames))
        bge_embs = self.bge_model.encode(spatial_texts, normalize_embeddings=True).astype('float32')
        for t_idx in range(len(spatial_texts)):
            emb = bge_embs[t_idx:t_idx+1]
            D, I = self.index_caption.search(emb, num_frames)
            for dist, idx in zip(D[0], I[0]):
                bge_all[t_idx, idx] = dist
        scores_dict['faiss_caption'] = np.max(bge_all, axis=0)

        # --- BM25: union bag-of-words ---
        combined_text = " ".join(spatial_texts)
        scores_dict['bm25_caption'] = self.bm25_caption.get_scores(combined_text.lower().split())

    def _select_diverse_topk(self, scores: np.ndarray, top_k: int, frames_per_video: int = 5) -> List[int]:
        order = np.argsort(scores)[::-1]
        video_frames = {}
        video_order = []
        for idx in order:
            vid = self.metadata[idx]['video_id']
            if vid not in video_frames:
                if len(video_order) >= top_k:
                    continue
                video_frames[vid] = []
                video_order.append(vid)
            if len(video_frames[vid]) < frames_per_video:
                video_frames[vid].append(int(idx))
        selected = []
        for vid in video_order:
            selected.extend(video_frames[vid])
        return selected

    def search(self, action_query: Dict, top_k: int = 10, frames_per_video: int = 5, visualize: bool = False, weights: Dict[str, float] = None, search_regions: Optional[List[str]] = None):
        if weights is None:
            weights = {'visual': 1.0, 'speech': 1.0, 'ocr': 1.0}

        num_frames = len(self.metadata)
        if num_frames == 0:
            return []

        region_mask = self._get_region_mask(search_regions)

        scores_dict = {
            'faiss_clip': np.zeros(num_frames), 'faiss_caption': np.zeros(num_frames), 'bm25_caption': np.zeros(num_frames),
            'faiss_speech': np.zeros(num_frames), 'bm25_speech': np.zeros(num_frames),
            'faiss_ocr': np.zeros(num_frames), 'bm25_ocr': np.zeros(num_frames)
        }

        active_modalities = []

        # --- Visual: Multi-level prompt ensembling ---
        spatial_texts = self._collect_spatial_texts(action_query)
        if spatial_texts:
            active_modalities.append('visual')
            self._score_visual_ensemble(spatial_texts, num_frames, scores_dict)

        if action_query.get('asr_text') and len(action_query['asr_text']) > 0:
            active_modalities.append('speech')
            asr_text = " ".join(action_query['asr_text'])

            query_emb = self.bge_model.encode([asr_text], normalize_embeddings=True).astype('float32')
            D, I = self.index_speech.search(query_emb, num_frames)
            for dist, idx in zip(D[0], I[0]):
                scores_dict['faiss_speech'][idx] = dist

            scores_dict['bm25_speech'] = self.bm25_speech.get_scores(asr_text.lower().split())

        if action_query.get('ocr_text') and len(action_query['ocr_text']) > 0:
            active_modalities.append('ocr')
            ocr_text = " ".join(action_query['ocr_text'])

            query_emb = self.bge_model.encode([ocr_text], normalize_embeddings=True).astype('float32')
            D, I = self.index_ocr.search(query_emb, self.index_ocr.ntotal)
            for dist, ocr_idx in zip(D[0], I[0]):
                frame_idx = self.ocr_to_frame_map[ocr_idx]
                if dist > scores_dict['faiss_ocr'][frame_idx]:
                    scores_dict['faiss_ocr'][frame_idx] = dist

            scores_dict['bm25_ocr'] = self.bm25_ocr.get_scores(ocr_text.lower().split())

        normalized_scores = {key: self._min_max_scale(scores) for key, scores in scores_dict.items()}

        score_vis = 0.5 * normalized_scores['faiss_clip'] + 0.5 * ((normalized_scores['faiss_caption'] + normalized_scores['bm25_caption']) / 2.0)
        score_speech = 0.6 * normalized_scores['faiss_speech'] + 0.4 * normalized_scores['bm25_speech']
        score_ocr = 0.35 * normalized_scores['faiss_ocr'] + 0.65 * normalized_scores['bm25_ocr']

        component_scores = {'visual': score_vis, 'speech': score_speech, 'ocr': score_ocr}
        eff_weights = self._fuse_weights(weights, active_modalities, component_scores)

        total_scores = np.zeros(num_frames)
        for m in active_modalities:
            total_scores += eff_weights.get(m, 0.0) * component_scores[m]

        # Áp dụng region mask
        total_scores = self._apply_region_mask(total_scores, region_mask)

        selected_indices = self._select_diverse_topk(total_scores, top_k, frames_per_video)
        final_return_ids = []

        if visualize:
            print(f"\n📊 CHI TIẾT ĐIỂM (SINGLE ACTION - TOP {len(selected_indices)}):")
            print(f"   Trọng số hiệu dụng: {eff_weights}")

        for rank, idx in enumerate(selected_indices, 1):
            meta = self.metadata[idx]
            final_return_ids.append([f"{meta['video_id']}_{meta['original_frame_idx']}"])
            if visualize:
                print(f"  [{rank}] {meta['video_id']}_{meta['original_frame_idx']} | ĐIỂM TỔNG: {total_scores[idx]:.4f} | Visual: {score_vis[idx]:.4f} | Speech: {score_speech[idx]:.4f} | OCR: {score_ocr[idx]:.4f}")

        return final_return_ids

    def search_temporal_events(self, actions: List[Dict], task_type: int, top_k: int = 5, frames_per_video: int = 5, visualize: bool = False, weights: Dict[str, float] = None, search_regions: Optional[List[str]] = None):
        if weights is None:
            weights = {'visual': 1.0, 'speech': 1.0, 'ocr': 1.0}

        num_frames = len(self.metadata)
        num_events = len(actions)

        if num_events == 0 or num_frames == 0:
            return []

        region_mask = self._get_region_mask(search_regions)

        all_events_scores = []
        all_detailed_scores = []
        all_effective_weights = []

        for action in actions:
            scores_dict = {
                'faiss_clip': np.zeros(num_frames), 'faiss_caption': np.zeros(num_frames), 'bm25_caption': np.zeros(num_frames),
                'faiss_speech': np.zeros(num_frames), 'bm25_speech': np.zeros(num_frames),
                'faiss_ocr': np.zeros(num_frames), 'bm25_ocr': np.zeros(num_frames)
            }
            active_modalities = []

            # --- Visual: Multi-level prompt ensembling ---
            spatial_texts = self._collect_spatial_texts(action)
            if spatial_texts:
                active_modalities.append('visual')
                self._score_visual_ensemble(spatial_texts, num_frames, scores_dict)

            if action.get('asr_text') and len(action['asr_text']) > 0:
                active_modalities.append('speech')
                vi_text = " ".join(action['asr_text'])
                bge_emb_vi = self.bge_model.encode([vi_text], normalize_embeddings=True).astype('float32')
                D_spch, I_spch = self.index_speech.search(bge_emb_vi, num_frames)
                for dist, idx in zip(D_spch[0], I_spch[0]):
                    scores_dict['faiss_speech'][idx] = dist
                scores_dict['bm25_speech'] = self.bm25_speech.get_scores(vi_text.lower().split())

            if action.get('ocr_text') and len(action['ocr_text']) > 0:
                active_modalities.append('ocr')
                vi_text_ocr = " ".join(action['ocr_text'])
                bge_emb_vi_ocr = self.bge_model.encode([vi_text_ocr], normalize_embeddings=True).astype('float32')
                D_ocr, I_ocr = self.index_ocr.search(bge_emb_vi_ocr, self.index_ocr.ntotal)
                for dist, ocr_idx in zip(D_ocr[0], I_ocr[0]):
                    frame_idx = self.ocr_to_frame_map[ocr_idx]
                    if dist > scores_dict['faiss_ocr'][frame_idx]:
                        scores_dict['faiss_ocr'][frame_idx] = dist
                scores_dict['bm25_ocr'] = self.bm25_ocr.get_scores(vi_text_ocr.lower().split())

            normalized_scores = {k: self._min_max_scale(v) for k, v in scores_dict.items()}
            all_detailed_scores.append(normalized_scores)

            score_vis = 0.5 * normalized_scores['faiss_clip'] + 0.5 * ((normalized_scores['faiss_caption'] + normalized_scores['bm25_caption']) / 2.0)
            score_speech = 0.6 * normalized_scores['faiss_speech'] + 0.4 * normalized_scores['bm25_speech']
            score_ocr = 0.35 * normalized_scores['faiss_ocr'] + 0.65 * normalized_scores['bm25_ocr']

            component_scores = {'visual': score_vis, 'speech': score_speech, 'ocr': score_ocr}
            eff_weights = self._fuse_weights(weights, active_modalities, component_scores)
            all_effective_weights.append(eff_weights)

            event_total_score = np.zeros(num_frames)
            for m in active_modalities:
                event_total_score += eff_weights.get(m, 0.0) * component_scores[m]

            # Áp dụng region mask cho từng event score
            event_total_score = self._apply_region_mask(event_total_score, region_mask)

            all_events_scores.append(event_total_score)

        A = np.array(all_events_scores)

        video_to_frames = {}
        for idx, meta in enumerate(self.metadata):
            vid = meta['video_id']
            if vid not in video_to_frames:
                video_to_frames[vid] = []
            video_to_frames[vid].append(idx)

        # Lọc video_to_frames theo region nếu có
        if region_mask is not None:
            filtered_video_to_frames = {}
            for vid, frame_indices in video_to_frames.items():
                filtered = [f for f in frame_indices if region_mask[f]]
                if filtered:
                    filtered_video_to_frames[vid] = filtered
            video_to_frames = filtered_video_to_frames

        sigma_min = 15.0
        sigma_ratio = 0.08
        video_chain_scores = {}
        video_best_chains = {}

        for vid, frame_indices in video_to_frames.items():
            n_frames = len(frame_indices)
            if n_frames < num_events:
                continue

            sigma = max(sigma_min, n_frames * sigma_ratio)

            if num_events == 1:
                best_f = int(np.argmax(A[0, frame_indices]))
                video_chain_scores[vid] = A[0, frame_indices[best_f]]
                video_best_chains[vid] = [frame_indices[best_f]]
                continue

            DP = np.zeros((num_events, n_frames))
            Trace = np.full((num_events, n_frames), -1, dtype=int)

            for f in range(n_frames):
                DP[0, f] = A[0, frame_indices[f]]

            for e in range(1, num_events):
                running_max = 0.0
                best_prev = -1
                for f in range(1, n_frames):
                    gap = frame_indices[f] - frame_indices[f - 1]
                    running_max *= np.exp(-gap / sigma)

                    if DP[e - 1, f - 1] >= running_max:
                        running_max = DP[e - 1, f - 1]
                        best_prev = f - 1

                    DP[e, f] = running_max * A[e, frame_indices[f]]
                    Trace[e, f] = best_prev

            best_f = int(np.argmax(DP[num_events - 1]))
            max_chain_score = DP[num_events - 1, best_f]

            if max_chain_score > 0 and Trace[num_events - 1, best_f] != -1:
                video_chain_scores[vid] = max_chain_score
                chain = []
                curr_f = best_f
                for e in range(num_events - 1, -1, -1):
                    chain.append(frame_indices[curr_f])
                    curr_f = Trace[e, curr_f]
                chain.reverse()
                video_best_chains[vid] = chain

        final_return_ids = []

        if task_type == 3:
            sorted_vids = sorted(video_chain_scores.keys(), key=lambda v: video_chain_scores[v], reverse=True)
            top_vids = sorted_vids[:top_k]

            if visualize:
                print(f"\n📊 CHI TIẾT ĐIỂM CHUỖI SỰ KIỆN (TASK 3 - TOP {len(top_vids)}):")

            for rank, vid in enumerate(top_vids, 1):
                chain = video_best_chains[vid]
                chain_score_normalized = video_chain_scores[vid] ** (1.0 / num_events)
                orig_ids = [self.metadata[f]['original_frame_idx'] for f in chain]
                final_return_ids.append([f"{vid}_{fid}" for fid in orig_ids])

                if visualize:
                    print(f"\n  [{rank}] Video: {vid} | Khung hình: {orig_ids} | CHUỖI SCORE TỔNG: {chain_score_normalized:.4f}")
                    for e, f_global in enumerate(chain):
                        orig_id = self.metadata[f_global]['original_frame_idx']
                        base_score = A[e, f_global]
                        det = all_detailed_scores[e]
                        v_score = 0.5 * det['faiss_clip'][f_global] + 0.5 * ((det['faiss_caption'][f_global] + det['bm25_caption'][f_global]) / 2.0)
                        s_score = 0.6 * det['faiss_speech'][f_global] + 0.4 * det['bm25_speech'][f_global]
                        o_score = 0.35 * det['faiss_ocr'][f_global] + 0.65 * det['bm25_ocr'][f_global]
                        print(f"       - Event {e+1} (Frame {orig_id}): Base Score = {base_score:.4f} | Visual = {v_score:.4f} | Speech = {s_score:.4f} | OCR = {o_score:.4f} | Trọng số = {all_effective_weights[e]}")

        else:
            alpha = 0.4

            vid_to_scaled_chain = {}
            for vid, raw_score in video_chain_scores.items():
                vid_to_scaled_chain[vid] = raw_score ** (1.0 / num_events)

            frame_final_scores = np.zeros(num_frames)
            for f in range(num_frames):
                vid = self.metadata[f]['video_id']
                base_score = np.max(A[:, f])
                chain_score = vid_to_scaled_chain.get(vid, 0.0)
                frame_final_scores[f] = alpha * base_score + (1.0 - alpha) * chain_score

            # Áp dụng region mask cho frame_final_scores
            frame_final_scores = self._apply_region_mask(frame_final_scores, region_mask)

            selected_indices = self._select_diverse_topk(frame_final_scores, top_k, frames_per_video)

            if visualize:
                print(f"\n📊 CHI TIẾT ĐIỂM KẾT HỢP (TASK {task_type} - TOP {len(selected_indices)}):")

            for rank, f in enumerate(selected_indices, 1):
                meta = self.metadata[f]
                vid = meta['video_id']
                orig_id = meta['original_frame_idx']
                final_return_ids.append([f"{vid}_{orig_id}"])

                if visualize:
                    blended = frame_final_scores[f]
                    base = np.max(A[:, f])
                    e_max = int(np.argmax(A[:, f]))
                    chain_s = vid_to_scaled_chain.get(vid, 0.0)

                    det = all_detailed_scores[e_max]
                    v_score = 0.5 * det['faiss_clip'][f] + 0.5 * ((det['faiss_caption'][f] + det['bm25_caption'][f]) / 2.0)
                    s_score = 0.6 * det['faiss_speech'][f] + 0.4 * det['bm25_speech'][f]
                    o_score = 0.35 * det['faiss_ocr'][f] + 0.65 * det['bm25_ocr'][f]

                    print(f"\n  [{rank}] Video: {vid}_{orig_id} | TOTAL BLENDED SCORE: {blended:.4f}")
                    print(f"       - Chi tiết Blend: (Base: {base:.4f} * {alpha} + Chain Score: {chain_s:.4f} * {1-alpha})")
                    print(f"       - Thành phần Event {e_max+1}: Visual = {v_score:.4f} | Speech = {s_score:.4f} | OCR = {o_score:.4f}")

        return final_return_ids

In [ ]:
# ===== CELL 4c - nạp dữ liệu tiết kiệm bộ nhớ + ĐO từng chặng =====
# Ghi đè _load_database. Index ra GIỐNG HỆT bản gốc: cùng thứ tự vector, cùng chuẩn hoá.
#
# Vì sao không chia khối gọi add() nhiều lần: IndexFlatIP lưu trong std::vector C++,
# vượt sức chứa là cấp phát GẤP ĐÔI rồi copy -> gọi add() 15 lần còn tốn hơn 1 lần.
# Cách đúng: cấp phát sẵn mảng đích một lần, đổ dữ liệu vào, add() đúng một lần.
import gc, json, os
import numpy as np
import faiss


def _mem_gb():
    for p in ("/sys/fs/cgroup/memory.stat", "/sys/fs/cgroup/memory/memory.stat"):
        try:
            st = dict(l.split()[:2] for l in open(p))
            return int(st.get("anon", 0)) / 1e9
        except OSError:
            pass
    return -1.0


def _log(tag):
    print(f"   [{_mem_gb():5.1f} GB] {tag}", flush=True)


def _load_database_lean(self):
    print(f"📥 Đang load: {self.db_path}  (bản tiết kiệm bộ nhớ)")
    _log("bắt đầu")

    # ---- LƯỢT 1: chỉ đọc json, đếm số vector, dựng metadata + corpus ----
    videos = []
    n_frames = n_ocr = 0
    for folder in sorted(os.listdir(self.db_path)):
        fp = os.path.join(self.db_path, folder)
        if not os.path.isdir(fp):
            continue
        for vid in sorted(os.listdir(fp)):
            vp = os.path.join(fp, vid)
            jp = os.path.join(vp, f"{vid}.json")
            if not (os.path.isdir(vp) and os.path.exists(jp)):
                continue
            if not all(os.path.exists(os.path.join(vp, f"{vid}{s}.npy"))
                       for s in ("", "_caption", "_speech", "_ocr")):
                print(f"   ⚠️ {vid}: thiếu npy, bỏ qua")
                continue

            with open(jp, encoding="utf-8") as f:
                frames = json.load(f)["frames"]
            ocr_npy = np.load(os.path.join(vp, f"{vid}_ocr.npy"), allow_pickle=True)

            counts = []
            for i, fr in enumerate(frames):
                fr["video_id"] = vid
                fr["folder_name"] = folder
                fr["global_idx"] = n_frames
                self.metadata.append(fr)
                self.corpus_caption.append(fr.get("visual_caption", "").lower().split())
                self.corpus_speech.append(fr.get("speech_text", "").lower().split())
                self.corpus_ocr.append(fr.get("ocr_text", "").replace("|", " ").lower().split())

                v = ocr_npy[i] if i < len(ocr_npy) else None
                k = v.shape[0] if getattr(v, "ndim", 0) == 2 and v.shape[1] == self.dim_bge else 0
                counts.append(k)
                for _ in range(k):
                    self.ocr_to_frame_map.append(n_frames)
                n_ocr += k
                n_frames += 1
            videos.append((vp, vid, len(frames), counts))
            del ocr_npy
    gc.collect()
    _log(f"lượt 1 xong: {n_frames:,} frame · {n_ocr:,} vector OCR · {len(videos)} video")

    # ---- LƯỢT 2: cấp phát SẴN từng ma trận, đổ vào, add MỘT lần ----
    def build(index, dim, n, fill, name):
        m = np.empty((n, dim), dtype="float32")   # cấp phát đúng một lần
        fill(m)
        faiss.normalize_L2(m)                      # chuẩn hoá theo từng dòng
        index.add(m)
        del m
        gc.collect()
        _log(f"{name}: {index.ntotal:,} vector")

    def filler(suffix, per_frame=False):
        def f(m):
            r = 0
            for vp, vid, nf, counts in videos:
                a = np.load(os.path.join(vp, f"{vid}{suffix}.npy"),
                            allow_pickle=per_frame)
                if per_frame:
                    for i in range(nf):
                        k = counts[i]
                        if k:
                            m[r:r + k] = a[i]
                            r += k
                else:
                    m[r:r + nf] = a[:nf]
                    r += nf
                del a
            assert r == len(m), f"đổ {r} dòng nhưng cấp phát {len(m)}"
        return f

    build(self.index_clip, self.dim_clip, n_frames, filler(""), "clip")
    build(self.index_caption, self.dim_bge, n_frames, filler("_caption"), "caption")
    build(self.index_speech, self.dim_bge, n_frames, filler("_speech"), "speech")
    build(self.index_ocr, self.dim_bge, n_ocr, filler("_ocr", True), "ocr")

    assert self.index_ocr.ntotal == len(self.ocr_to_frame_map), "index_ocr lệch ocr_to_frame_map"
    print(f"✅ Hoàn tất load {len(self.metadata)} frames vào hệ thống!")


VideoSearchEngine._load_database = _load_database_lean
print("✅ đã ghi đè _load_database. Bỏ cell này + restart là quay về bản gốc.")

In [ ]:
# ===== CELL 4d — tầng trả kết quả theo task (nguyên văn searcher.py) =====
# Trả về list dict: {"loại task": "task N", "kết quả": [chuỗi đúng định dạng BTC]}
#   task 1: "video_id, frame_id"
#   task 2: "video_id, frame_id, question"
#   task 3: "video_id, frame_id1, frame_id2, ..., frame_idn"


def process_queries(queries: List[Dict], db_path: str = None, top_k: int = 5,
                    frames_per_video: int = 5, visualize: bool = False,
                    search_regions: Optional[List[str]] = None, engine=None) -> List[Dict]:
    """
    Hàm xử lý danh sách các truy vấn, tự động định tuyến, in log trực quan (nếu bật) và format kết quả.
    - top_k: Số lượng kết quả tốt nhất trả về cho mỗi query.
    - search_regions: Danh sách tên thư mục (vd: ['Videos_L21_a', 'Videos_L22_a'])
                     để thu hẹp vùng tìm kiếm. None = tìm toàn bộ.
    """
    if engine is None:
        engine = VideoSearchEngine(db_path=db_path)
    final_output = []

    for i, q in enumerate(queries, 1):
        user_prompt = q.get("prompt", "")
        explicit_task_type = q.get("task_type", None)
        custom_weights = q.get("weights", {'visual': 1.0, 'speech': 1.0, 'ocr': 1.0})
        custom_frames_per_video = q.get("frames_per_video", frames_per_video)
        query_regions = q.get("search_regions", search_regions)

        if visualize:
            print("\n\n" + "=" * 80)
            print(f"🚀 [TEST CASE {i}]")
            print(f"👤 NGƯỜI DÙNG NHẬP: {user_prompt}")
            if explicit_task_type:
                print(f"📌 ÉP BUỘC TASK TYPE: {explicit_task_type}")
            if query_regions:
                print(f"📂 VÙNG TÌM KIẾM: {query_regions}")
            print(f"🎯 TRẢ VỀ: Top {top_k} kết quả")
            print("=" * 80)
            print("\n🧠 Đang gọi LLM để phân tích truy vấn...")

        try:
            parsed_query = analyze_prompt(user_prompt, explicit_task_type=explicit_task_type)

            if visualize:
                print("\n✅ KẾT QUẢ LLM PHÂN TÍCH ĐÃ TRẢ VỀ (DICT/JSON):")
                print(json.dumps(parsed_query, ensure_ascii=False, indent=2))
        except Exception as e:
            if visualize:
                print(f"❌ Lỗi khi phân tích prompt hoặc chưa định nghĩa hàm analyze_prompt: {e}")

            parsed_query = {"task_type": explicit_task_type or 1, "actions": [{"spatial_context_detailed": user_prompt}], "questions": []}

        if not parsed_query:
            continue

        task_type = parsed_query.get('task_type', explicit_task_type or 1)
        actions = parsed_query.get('actions', [])
        questions = parsed_query.get('questions', [])

        formatted_results = []

        if visualize:
            print("\n🔍 ĐANG TIẾN HÀNH TÌM KIẾM TRONG DATABASE...")

        if len(actions) > 1:
            raw_results = engine.search_temporal_events(
                actions=actions,
                task_type=task_type,
                top_k=top_k,
                frames_per_video=custom_frames_per_video,
                visualize=visualize,
                weights=custom_weights,
                search_regions=query_regions
            )

            for res in raw_results:
                if not res:
                    continue
                vid = res[0].rsplit('_', 1)[0]
                frame_ids = [item.rsplit('_', 1)[1] for item in res]

                if task_type == 3:
                    formatted_string = f"{vid}, " + ", ".join(frame_ids)
                elif task_type == 2:
                    answer = questions[0] if questions else "Unknown Answer"
                    formatted_string = f"{vid}, {frame_ids[-1]}, {answer}"
                else:
                    formatted_string = f"{vid}, {frame_ids[0]}"

                formatted_results.append(formatted_string)

        else:
            single_action = actions[0] if len(actions) > 0 else {}
            raw_results = engine.search(
                action_query=single_action,
                top_k=top_k,
                frames_per_video=custom_frames_per_video,
                visualize=visualize,
                weights=custom_weights,
                search_regions=query_regions
            )

            for res in raw_results:
                if not res:
                    continue
                vid = res[0].rsplit('_', 1)[0]
                frame_id = res[0].rsplit('_', 1)[1]

                if task_type == 2:
                    answer = questions[0] if questions else "Unknown Answer"
                    formatted_string = f"{vid}, {frame_id}, {answer}"
                else:
                    formatted_string = f"{vid}, {frame_id}"

                formatted_results.append(formatted_string)

        if visualize:
            print(f"\n🎯 Danh sách {top_k} chuỗi kết quả đã format (Để submit):")
            for fr in formatted_results:
                print("   -", fr)

        final_output.append({
            "loại task": f"task {task_type}",
            "kết quả": formatted_results
        })

    return final_output

In [ ]:
%%writefile frame_resolver.py
"""
Nối mixed_database với keyframe/ảnh của BTC.

Bối cảnh (đã đo trên cả 873 video, xem map.md):
  - original_frame_idx của mixed_database LÀ frame thật trong video gốc
    (timestamp_sec == original_frame_idx / fps, đúng tuyệt đối trên mọi video).
  - Nó lệch tối đa ĐÚNG 1 FRAME so với frame_idx của BTC - cùng tập keyframe,
    khác cách làm tròn. 87.4% trùng khít, còn lại lệch 1.
  - Vì vậy: tra sang keyframe BTC gần nhất là an toàn, và nên NỘP frame_idx của BTC
    thay vì original_frame_idx (miễn phí, và bỏ được rủi ro lệch 1 frame có hệ thống).

Cách dùng:
    r = FrameResolver()
    hit = r.resolve("L21_V001", 711)
    hit.submit_frame_id   -> nộp bài
    hit.keyframe_path     -> ảnh hiển thị
    hit.pts_time          -> ffmpeg -ss (cắt clip TRAKE)
"""

from __future__ import annotations

import glob
import os
import pickle
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import numpy as np
import pandas as pd

import glob as _glob

def _auto(pattern: str, depth_max: int = 6) -> str:
    """Dò thư mục dữ liệu BTC, chịu được việc đổi slug dataset."""
    for d in range(1, depth_max + 1):
        hits = _glob.glob("/kaggle/input/" + "*/" * d + pattern)
        if hits:
            return hits[0]
    return ""

MAP_KEYFRAMES_DIR = _auto("map-keyframes*/map-keyframes") or _auto("map-keyframes")

# Dò Keyframes ĐỘC LẬP thay vì suy từ MAP_KEYFRAMES_DIR - suy chuỗi kiểu đó rất dễ
# cắt lố một cấp thư mục, và khi cắt lố thì /search vẫn chạy còn /image lặng lẽ 404.
_kf_one = _auto("Keyframes_*/keyframes")
KEYFRAMES_GLOB = (os.path.dirname(os.path.dirname(_kf_one)) + "/Keyframes_*/keyframes"
                  if _kf_one else "")

MAX_EXPECTED_GAP = 2


@dataclass(frozen=True)
class ResolvedFrame:
    video_id: str
    submit_frame_id: int   # frame_idx của BTC - GIÁ TRỊ ĐEM NỘP
    n: int                 # số thứ tự keyframe, quyết định tên file ảnh
    keyframe_path: str     # ảnh để hiển thị lên web
    pts_time: float        # giây, dùng cho `ffmpeg -ss`
    fps: float
    gap: int               # lệch bao nhiêu frame so với giá trị mixed_database đưa vào


class FrameResolver:
    """Nạp map-keyframes một lần (có cache đĩa + đọc song song), tra O(log n) mỗi frame."""

    def __init__(self, map_dir: str = MAP_KEYFRAMES_DIR, keyframes_glob: str = KEYFRAMES_GLOB):
        self._frames: dict[str, np.ndarray] = {}
        self._n: dict[str, np.ndarray] = {}
        self._pts: dict[str, np.ndarray] = {}
        self._fps: dict[str, float] = {}
        self._kf_dir: dict[str, str] = {}

        # 1. Dò cache đĩa trước (trong /kaggle/input hoặc /kaggle/working)
        cache_paths = [
            "/kaggle/input/aic_index_cache/frame_resolver_cache.pkl",
            "/kaggle/working/aic_index_cache/frame_resolver_cache.pkl",
            "/kaggle/working/frame_resolver_cache.pkl",
        ]
        loaded_from_cache = False
        for cp in cache_paths:
            if os.path.isfile(cp):
                try:
                    t0 = time.time()
                    with open(cp, "rb") as fh:
                        cached = pickle.load(fh)
                        self._frames = cached["_frames"]
                        self._n = cached["_n"]
                        self._pts = cached["_pts"]
                        self._fps = cached["_fps"]
                        self._kf_dir = cached["_kf_dir"]
                        loaded_from_cache = True
                        print(f"⚡ FrameResolver: Đọc thẳng cache {cp} trong {time.time()-t0:.2f}s ({len(self._frames)} video map · {len(self._kf_dir)} thư mục keyframe)")
                        break
                except Exception:
                    pass

        if not loaded_from_cache:
            t0 = time.time()
            csv_files = glob.glob(os.path.join(map_dir, "*.csv"))

            def _read_csv(csv_path):
                video_id = os.path.basename(csv_path)[: -len(".csv")]
                df = pd.read_csv(csv_path).sort_values("frame_idx")
                return (
                    video_id,
                    df["frame_idx"].astype(int).to_numpy(),
                    df["n"].astype(int).to_numpy(),
                    df["pts_time"].astype(float).to_numpy(),
                    float(df["fps"].iloc[0])
                )

            with ThreadPoolExecutor(max_workers=32) as ex:
                for video_id, frames, n, pts, fps in ex.map(_read_csv, csv_files):
                    self._frames[video_id] = frames
                    self._n[video_id] = n
                    self._pts[video_id] = pts
                    self._fps[video_id] = fps

            # Keyframes nằm rải ở Keyframes_L21/, Keyframes_L22/... - lập chỉ mục một lần
            for root in glob.glob(keyframes_glob):
                for vdir in glob.glob(os.path.join(root, "*")):
                    if os.path.isdir(vdir):
                        self._kf_dir[os.path.basename(vdir)] = vdir

            print(f"FrameResolver: Nạp song song xong trong {time.time()-t0:.2f}s ({len(self._frames)} video map · {len(self._kf_dir)} thư mục keyframe)")

            # Lưu cache vào /kaggle/working để các lần sau đọc tức thì trong 0.01s
            try:
                out_dir = "/kaggle/working/aic_index_cache"
                os.makedirs(out_dir, exist_ok=True)
                cache_file = os.path.join(out_dir, "frame_resolver_cache.pkl")
                with open(cache_file, "wb") as fh:
                    pickle.dump({
                        "_frames": self._frames,
                        "_n": self._n,
                        "_pts": self._pts,
                        "_fps": self._fps,
                        "_kf_dir": self._kf_dir
                    }, fh, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"💾 Đã lưu cache FrameResolver tại {cache_file}")
            except Exception:
                pass

        if not self._frames:
            raise FileNotFoundError(
                f"Không đọc được map-keyframes nào từ {map_dir!r}.\n"
                f"Đã attach dataset chứa map-keyframes chưa?"
            )
        if not self._kf_dir:
            raise FileNotFoundError(
                f"Không thấy thư mục keyframe nào khớp {keyframes_glob!r}.\n"
                f"/search sẽ chạy nhưng /image sẽ 404 -> web không có ảnh."
            )

    def resolve(self, video_id: str, original_frame_idx: int) -> ResolvedFrame:
        """Tra frame của mixed_database sang keyframe BTC gần nhất."""
        if video_id not in self._frames:
            raise KeyError(f"{video_id}: không có map-keyframes")

        frames = self._frames[video_id]
        pos = int(np.searchsorted(frames, original_frame_idx))

        # chọn bên trái hay bên phải, cái nào gần hơn
        best, best_gap = None, None
        for p in (pos - 1, pos):
            if 0 <= p < len(frames):
                g = abs(int(frames[p]) - int(original_frame_idx))
                if best_gap is None or g < best_gap:
                    best, best_gap = p, g

        n = int(self._n[video_id][best])
        kf_dir = self._kf_dir.get(video_id)

        return ResolvedFrame(
            video_id=video_id,
            submit_frame_id=int(frames[best]),
            n=n,
            keyframe_path=os.path.join(kf_dir, f"{n:03d}.jpg") if kf_dir else "",
            pts_time=float(self._pts[video_id][best]),
            fps=self._fps[video_id],
            gap=int(best_gap),
        )

    def resolve_many(self, pairs: list[tuple[str, int]]) -> list[ResolvedFrame]:
        return [self.resolve(v, f) for v, f in pairs]


In [ ]:
%%writefile api_server.py
"""
API bọc quanh `VideoSearchEngine` GỐC - phơi ra ngoài bằng cloudflared.

KHÔNG sửa một dòng nào trong searcher.py. Nó chỉ:
  1. nhận `engine` bạn đã tạo ở cell 5
  2. gọi `analyze_prompt()` + `engine.search()` y như notebook vẫn làm
  3. tra `original_frame_idx` sang keyframe BTC để lấy ảnh (FrameResolver)
  4. trả JSON cho web

Vì sao cần FrameResolver: engine gốc trả `original_frame_idx` (vd 711), nhưng file ảnh BTC
đặt tên theo số thứ tự keyframe (`007.jpg`). Hai hệ đánh số khác nhau - đã đo trên cả 873
video, lệch tối đa 1 frame. Không tra thì web hiện ẢNH SAI mà không báo lỗi.
FrameResolver chỉ TRA CỨU, không tham gia tìm kiếm, không đụng tới thứ hạng.
"""

import hashlib
import io
import json
import os
import re
import subprocess
import threading
import time

TEAM_KEY = "aicdeepbyte"   # key chung của đội - đổi chuỗi này là mọi key cũ hết hiệu lực


def make_key(salt: str = TEAM_KEY) -> str:
    """Key CHUNG CỦA ĐỘI - KHÔNG suy từ tài khoản người chạy.

    Vì sao: `aic.verse.id.vn` là MỘT tunnel dùng chung. Ai bật notebook cũng đăng ký
    connector vào đó, và Cloudflare chia tải ngẫu nhiên giữa các connector. Nếu key
    suy từ username thì:
      - hai người cùng bật -> key của A chỉ đúng ~50% request -> 401 nhấp nháy
      - A tắt, B bật       -> cả đội phải đổi key

    Key chung thì ai chạy cũng vậy, và hai người cùng chạy cũng vô hại (cùng dữ liệu,
    cùng index -> kết quả giống hệt nhau).

    Ưu tiên Kaggle Secret `AIC_ACCESS_KEY` nếu muốn đổi mà không sửa code.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        k = UserSecretsClient().get_secret("AIC_ACCESS_KEY")
        if k:
            return k.strip()
    except Exception:
        pass
    return salt


CLOUDFLARED_URL = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
)


def build_app(engine, analyze_fn=None, resolver=None, access_key=None):
    """engine: VideoSearchEngine gốc. analyze_fn: analyze_prompt (tuỳ chọn)."""
    from fastapi import FastAPI, HTTPException, Query, Request
    from fastapi.middleware.cors import CORSMiddleware
    from fastapi.middleware.gzip import GZipMiddleware
    from fastapi.responses import FileResponse, Response
    from pydantic import BaseModel
    from typing import Optional
    import glob

    from frame_resolver import FrameResolver

    resolver = resolver or FrameResolver()

    app = FastAPI(title="AIC Search API")
    app.add_middleware(CORSMiddleware, allow_origins=["*"],
                       allow_methods=["*"], allow_headers=["*"])
    app.add_middleware(GZipMiddleware, minimum_size=1000)

    if access_key:
        from fastapi.responses import JSONResponse

        @app.middleware("http")
        async def check_key(request, call_next):
            # Ảnh nạp bằng <img src> nên KHÔNG gửi được header -> phải nhận cả ?key=
            got = (request.headers.get("x-aic-key")
                   or request.query_params.get("key"))
            if request.method == "OPTIONS" or got == access_key:
                return await call_next(request)
            # Middleware này chạy NGOÀI CORSMiddleware (đăng ký sau -> bọc ngoài), nên
            # response 401 trả thẳng từ đây KHÔNG được CORS gắn header. Trình duyệt khi
            # đó chặn luôn và báo "Failed to fetch" thay vì "401" -> người dùng tưởng
            # backend chết, thật ra chỉ sai key. Phải tự gắn header.
            return JSONResponse(
                {"detail": "Sai key hoặc thiếu key"},
                status_code=401,
                headers={
                    "Access-Control-Allow-Origin": request.headers.get("origin", "*"),
                    "Access-Control-Allow-Headers": "*",
                    "Access-Control-Allow-Methods": "*",
                },
            )

    # (video_id, original_frame_idx) -> metadata, để lấy caption/speech/ocr làm bằng chứng
    meta_by_id = {(m["video_id"], int(m["original_frame_idx"])): m for m in engine.metadata}

    # Cache danh sách đường dẫn mp4 để không phải glob lại
    video_path = {}
    vpath_cache = "/kaggle/working/aic_index_cache/video_path.pkl"
    if os.path.isfile(vpath_cache):
        try:
            with open(vpath_cache, "rb") as fh:
                video_path = pickle.load(fh)
        except Exception:
            pass
    if not video_path and resolver._kf_dir:
        base_vid_dir = os.path.dirname(list(resolver._kf_dir.values())[0]).rsplit("/Keyframes_", 1)[0]
        for d in glob.glob(base_vid_dir + "/Videos_*/video"):
            for f in glob.glob(os.path.join(d, "*.mp4")):
                video_path[os.path.basename(f)[:-4]] = f
        try:
            os.makedirs(os.path.dirname(vpath_cache), exist_ok=True)
            with open(vpath_cache, "wb") as fh:
                pickle.dump(video_path, fh, protocol=pickle.HIGHEST_PROTOCOL)
        except Exception:
            pass

    class SearchReq(BaseModel):
        prompt: str = ""
        spatial_context: list = []
        asr_text: list = []
        ocr_text: list = []
        top_k: int = 100
        weights: Optional[dict] = None
        # 1 = KIS, 2 = Q&A, 3 = TRAKE. Web biết chắc loại task nên ép thẳng,
        # không để LLM tự đoán - nó đoán nhầm thì cả truy vấn hỏng.
        task_type: Optional[int] = None
        search_regions: Optional[list] = None

    @app.get("/health")
    def health():
        return {"ok": True, "frames": len(engine.metadata),
                "videos": len({m["video_id"] for m in engine.metadata}),
                "llm": analyze_fn is not None}

    @app.post("/search")
    def search(req: SearchReq):
        """Định tuyến theo task, y như process_queries() trong searcher.py.

        Chữ ký engine đã ĐỔI ở bản mới:
          search(action_query=..)            <- một action, cho Task 1/2
          search_temporal_events(actions=..) <- chuỗi action, cho Task 3 (quy hoạch động)
        Cả hai trả về list các LIST id, không phải list chuỗi phẳng như bản cũ.
        """
        w = req.weights or {"visual": 1.0, "speech": 1.0, "ocr": 1.0}

        if req.prompt.strip() and analyze_fn is not None:
            parsed = analyze_fn(req.prompt, explicit_task_type=req.task_type)
        else:
            # web tự điền 3 trường -> dựng một action giả cho đúng chữ ký mới
            parsed = {
                "task_type": req.task_type or 1,
                "questions": [],
                "actions": [{
                    "spatial_context": req.spatial_context,
                    "asr_text": req.asr_text,
                    "ocr_text": req.ocr_text,
                }],
            }

        task_type = req.task_type if req.task_type in [1, 2, 3] else parsed.get("task_type", 1)
        parsed["task_type"] = task_type
        actions = parsed.get("actions", [])
        questions = parsed.get("questions", [])

        # Định tuyến y hệt process_queries() trong searcher.py: nhiều action đi chain
        # path, một action đi search() cũ. Chain path KHÔNG chạy được với num_events
        # == 1 vì Trace[0] luôn là -1 nên guard loại sạch mọi video.
        if len(actions) > 1:
            raw = engine.search_temporal_events(
                actions=actions, task_type=task_type,
                top_k=req.top_k, visualize=False, weights=w,
                search_regions=req.search_regions)
        else:
            raw = engine.search(
                action_query=actions[0] if actions else {},
                top_k=req.top_k, visualize=False, weights=w,
                search_regions=req.search_regions)

        # ---- chuỗi đúng định dạng BTC, giống hệt process_queries ----
        formatted = []
        for res in raw:
            if not res:
                continue
            vid = res[0].rsplit("_", 1)[0]
            fids = [x.rsplit("_", 1)[1] for x in res]
            if task_type == 3:
                formatted.append(f"{vid}, " + ", ".join(fids))
            elif task_type == 2:
                q = questions[0] if questions else "Unknown Question"
                formatted.append(f"{vid}, {fids[0]}, {q}")
            else:
                formatted.append(f"{vid}, {fids[0]}")

        # ---- candidate phẳng cho lưới ảnh trên web ----
        out = []
        for gi, res in enumerate(raw):
            for s in (res or []):
                video_id, _, fs = s.rpartition("_")
                orig = int(fs)
                hit = resolver.resolve(video_id, orig)
                m = meta_by_id.get((video_id, orig), {})
                sc = getattr(engine, "last_scores", {}).get(f"{video_id}_{orig}") or {}
                # Task 1/2: base + chain (và total đã trộn). Task 3: chỉ chain.
                out.append({
                    "rank": len(out) + 1,
                    "score": sc.get("total", sc.get("chain")),
                    "base_score": sc.get("base"),
                    "chain_score": sc.get("chain"),
                    "group": gi,                      # TRAKE: các frame cùng group = một đáp án
                    "video_id": video_id,
                    "frame_id": hit.submit_frame_id,
                    "orig_frame_idx": orig,
                    "keyframe_n": hit.n,
                    "pts_time": hit.pts_time,
                    "fps": float(hit.fps),
                    "image": f"/image?video_id={video_id}&n={hit.n}",
                    "caption": (m.get("visual_caption") or "")[:300],
                    "speech": (m.get("speech_text") or "")[:200],
                    "ocr": (m.get("ocr_text") or "")[:200],
                })

        return {
            "count": len(out),
            "task_type": task_type,
            "parsed": parsed,
            "formatted": formatted,     # đúng định dạng nộp bài
            "results": out,             # để web dựng lưới ảnh
        }

    # Cache thumbnail trong RAM. Không có nó thì mỗi request phải mở JPEG + resize +
    # encode lại; 100 ảnh cùng lúc là tràn hàng đợi. 3000 thumbnail 512px ~ 100 MB.
    from collections import OrderedDict
    _thumb = OrderedDict()
    _THUMB_MAX = 3000

    class ThumbReq(BaseModel):
        items: list = []
        w: int = 320

    @app.get("/image")
    def image(video_id: str, n: int, w: int = Query(512, ge=64, le=1280)):
        key = (video_id, n, w)
        hit = _thumb.get(key)
        if hit is not None:
            _thumb.move_to_end(key)
            return Response(hit, media_type="image/jpeg",
                            headers={"Cache-Control": "public, max-age=86400"})

        from PIL import Image
        kf_dir = resolver._kf_dir.get(video_id)
        if not kf_dir:
            raise HTTPException(404, f"không có keyframe của {video_id}")
        path = os.path.join(kf_dir, f"{n:03d}.jpg")
        if not os.path.exists(path):
            raise HTTPException(404, f"không thấy {path}")

        im = Image.open(path)
        im.thumbnail((w, w))
        buf = io.BytesIO()
        im.convert("RGB").save(buf, "JPEG", quality=80)
        data = buf.getvalue()

        _thumb[key] = data
        while len(_thumb) > _THUMB_MAX:
            _thumb.popitem(last=False)

        return Response(data, media_type="image/jpeg",
                        headers={"Cache-Control": "public, max-age=86400"})


    _frame_cache = OrderedDict()
    _FRAME_MAX = 600

    # (video_id, original_frame_idx) -> chỉ số trong engine.metadata. index_clip xếp
    # thẳng hàng 1:1 với metadata (xem search(): scores[idx] = dist), nên chỉ số này
    # dùng luôn được cho FAISS. Dựng một lần, ~177k mục.
    _row_of = {(m["video_id"], int(m["original_frame_idx"])): i
               for i, m in enumerate(engine.metadata)}

    @app.get("/similar")
    def similar(video_id: str, frame_id: int, top_k: int = Query(100, ge=1, le=500)):
        """Tìm frame GIỐNG VỀ HÌNH ẢNH với một frame cho trước.

        `frame_id` ở đây là `orig_frame_idx` mà /search trả về, không phải frame_id
        của BTC - hai số này lệch nhau tối đa 1, và metadata đánh chỉ mục theo cái đầu.

        Vì sao cần: gõ chữ rồi hy vọng CLIP hiểu là cách duy nhất đang có. Với cảnh
        khó tả bằng lời (góc máy, bố cục, một món đồ lạ) thì tìm được MỘT frame tàm
        tạm rồi xoay sang tìm ảnh giống nhanh hơn nhiều so với nghĩ thêm câu mô tả.
        """
        row = _row_of.get((video_id, frame_id))
        if row is None:
            raise HTTPException(404, f"không có frame {video_id}_{frame_id} trong index")

        vec = engine.index_clip.reconstruct(int(row)).reshape(1, -1)
        # +1 vì chính nó luôn đứng đầu (cosine = 1), lát bỏ đi
        D, I = engine.index_clip.search(vec, int(top_k) + 1)

        out = []
        for dist, idx in zip(D[0], I[0]):
            if idx < 0 or int(idx) == row:
                continue
            meta = engine.metadata[int(idx)]
            vid = meta["video_id"]
            orig = int(meta["original_frame_idx"])
            hit = resolver.resolve(vid, orig)
            m = meta_by_id.get((vid, orig), {})
            out.append({
                "rank": len(out) + 1,
                "score": round(float(dist), 4),
                "base_score": round(float(dist), 4),
                "chain_score": None,
                # Mỗi kết quả là MỘT đáp án độc lập, nên phải có group riêng. Gán chung
                # một số thì lưới coi cả 100 frame là một đáp án TRAKE 100 action.
                "group": len(out),
                "video_id": vid,
                "frame_id": hit.submit_frame_id,
                "orig_frame_idx": orig,
                "keyframe_n": hit.n,
                "pts_time": hit.pts_time,
                "fps": float(hit.fps),
                "image": f"/image?video_id={vid}&n={hit.n}",
                "caption": (m.get("visual_caption") or "")[:300],
                "speech": (m.get("speech_text") or "")[:200],
                "ocr": (m.get("ocr_text") or "")[:200],
            })
            if len(out) >= top_k:
                break

        return {"count": len(out), "parsed": {}, "results": out}

    @app.get("/frame")
    def frame(video_id: str, frame_id: int, w: int = Query(512, ge=64, le=1280)):
        """Ảnh của MỘT frame bất kỳ, trích thẳng từ video gốc.

        Khác `/image`: cái đó chỉ trả được ảnh KEYFRAME có sẵn trên đĩa. Sau khi
        người dùng rà thanh tua sang frame khác, `/image` vẫn trả keyframe cũ nên
        ảnh xem trước không khớp với frame sắp nộp - đúng thứ gây nhầm khi đối chiếu.

        Chậm hơn `/image` vì phải gọi ffmpeg, nên chỉ dùng cho danh sách đáp án
        (những frame THẬT SỰ đem nộp), đừng dùng cho lưới 100 kết quả.
        """
        key = (video_id, frame_id, w)
        hit = _frame_cache.get(key)
        if hit is not None:
            _frame_cache.move_to_end(key)
            return Response(hit, media_type="image/jpeg",
                            headers={"Cache-Control": "public, max-age=86400"})

        src = video_path.get(video_id)
        if not src:
            raise HTTPException(404, f"không thấy video {video_id}")

        fps = float(resolver._fps.get(video_id) or 25.0)
        t = max(0.0, frame_id / fps)
        # -ss TRƯỚC -i + -accurate_seek: nhanh mà vẫn đúng frame. Đặt -ss sau -i thì
        # ffmpeg giải mã từ đầu video, mỗi ảnh mất hàng chục giây.
        p = subprocess.run(
            ["ffmpeg", "-nostdin", "-threads", "2", "-an", "-sn", "-dn",
             "-accurate_seek", "-ss", f"{t:.5f}", "-i", src,
             "-frames:v", "1", "-vf", f"scale={w}:-2", "-q:v", "4", "-f", "image2", "-"],
            capture_output=True,
        )
        if p.returncode != 0 or not p.stdout:
            raise HTTPException(500, f"ffmpeg không trích được frame {frame_id}")
        data = p.stdout

        _frame_cache[key] = data
        while len(_frame_cache) > _FRAME_MAX:
            _frame_cache.popitem(last=False)
        return Response(data, media_type="image/jpeg",
                        headers={"Cache-Control": "public, max-age=86400"})

    @app.post("/thumbs")
    def thumbs(req: ThumbReq):
        """Gộp NHIỀU thumbnail vào MỘT response, xử lý đa luồng song song."""
        import base64
        from PIL import Image
        from concurrent.futures import ThreadPoolExecutor

        def _process(it):
            video_id, n = it[0], int(it[1])
            key = (video_id, n, req.w)
            data = _thumb.get(key)
            if data is not None:
                return f"{video_id}-{n}", base64.b64encode(data).decode()
            kf_dir = resolver._kf_dir.get(video_id)
            if not kf_dir:
                return f"{video_id}-{n}", None
            path = os.path.join(kf_dir, f"{n:03d}.jpg")
            if not os.path.exists(path):
                return f"{video_id}-{n}", None
            try:
                im = Image.open(path)
                im.thumbnail((req.w, req.w), Image.Resampling.BILINEAR)
                buf = io.BytesIO()
                im.convert("RGB").save(buf, "JPEG", quality=75, optimize=False)
                data = buf.getvalue()
                _thumb[key] = data
                while len(_thumb) > _THUMB_MAX:
                    _thumb.popitem(last=False)
                return f"{video_id}-{n}", base64.b64encode(data).decode()
            except Exception:
                return f"{video_id}-{n}", None

        out = {}
        with ThreadPoolExecutor(max_workers=8) as ex:
            for k, b64 in ex.map(_process, req.items[:200]):
                if b64 is not None:
                    out[k] = b64
        return {"thumbs": out}

    def _clip_window(video_id, frame_id, seconds):
        """Mốc clip tính bằng SỐ NGUYÊN FRAME, không đi qua pts_time.

        pts_time trong map-keyframes làm tròn 1 chữ số thập phân: frame 997 @25fps thật
        ra ở 39.88s nhưng CSV ghi 39.9 — lệch 0.02s = NỬA FRAME, đủ gây off-by-one khi
        làm tròn. Đi từ frame_id thì start rơi ĐÚNG biên frame, nên frame ở giây 0 của
        clip đúng bằng first_frame.
        """
        hit = resolver.resolve(video_id, frame_id)
        fps = float(hit.fps)
        n = int(round(seconds * fps))
        half = n // 2
        first = max(0, hit.submit_frame_id - half)
        return hit, fps, n, first, first / fps

    @app.get("/clipinfo")
    def clipinfo(video_id: str, frame_id: int, seconds: float = 5.0):
        hit, fps, n, first, start = _clip_window(video_id, frame_id, seconds)
        return {
            "video_id": video_id, "fps": fps,
            "center_frame": hit.submit_frame_id,
            # n / pts_time / gap: cho tab "Nhảy tới frame" trên web. /image và /thumbs
            # đều nhận `n` chứ không nhận frame_id, nên thiếu n là không lấy được các
            # keyframe lân cận. `gap` để giao diện nói rõ đã bắt về keyframe nào, thay
            # vì âm thầm đổi số dưới tay người dùng.
            "n": hit.n,
            "pts_time": hit.pts_time,
            "gap": hit.gap,
            "start_sec": round(start, 6), "seconds": seconds,
            "first_frame": first, "last_frame": first + n - 1, "n_frames": n,
        }

    WS_DIR = "/kaggle/working/ws"

    def _ws_path(name: str) -> str:
        """Chặn path traversal: `name` đi thẳng từ URL nên không được tin.

        Chỉ cho chữ, số, gạch ngang, gạch dưới. Mọi thứ khác bị loại, và tên rỗng
        sau khi lọc thì từ chối luôn - nếu không `../../` sẽ ghi đè ra ngoài WS_DIR.
        """
        safe = "".join(ch for ch in name if ch.isalnum() or ch in "-_")[:40]
        if not safe:
            raise HTTPException(400, "tên không hợp lệ")
        os.makedirs(WS_DIR, exist_ok=True)
        return os.path.join(WS_DIR, safe + ".json")

    @app.get("/ws")
    def ws_list():
        """Danh sách workspace đang có trên máy này."""
        os.makedirs(WS_DIR, exist_ok=True)
        out = []
        for f in sorted(os.listdir(WS_DIR)):
            if not f.endswith(".json"):
                continue
            p = os.path.join(WS_DIR, f)
            out.append({"name": f[:-5], "size": os.path.getsize(p),
                        "saved_at": int(os.path.getmtime(p))})
        return {"items": out}

    @app.get("/ws/{name}")
    def ws_get(name: str):
        p = _ws_path(name)
        if not os.path.exists(p):
            raise HTTPException(404, f"không có workspace {name}")
        return Response(open(p, "rb").read(), media_type="application/json")

    @app.put("/ws/{name}")
    async def ws_put(name: str, request: Request):
        """Đẩy workspace lên. Ghi đè bản cùng tên - mỗi người một tên riêng.

        LƯU Ý: /kaggle/working mất khi phiên kết thúc. Đây là chỗ trao đổi nhanh,
        KHÔNG phải nơi lưu trữ. Vẫn phải bấm "Lưu workspace" để giữ bản trên máy.
        """
        raw = await request.body()
        if len(raw) > 20 * 1024 * 1024:
            raise HTTPException(413, "workspace quá lớn")
        try:
            json.loads(raw)          # từ chối sớm, đừng để người khác tải về rồi mới vỡ
        except ValueError:
            # CHỈ bắt lỗi phân tích JSON. `except Exception` ở đây từng nuốt một
            # NameError (thiếu `import json`) và báo thành "sai JSON" - sai hoàn toàn,
            # mất cả buổi mới tìm ra. Lỗi khác cứ để nó nổ thành 500 cho đúng bản chất.
            raise HTTPException(400, "không phải JSON hợp lệ")
        # Ghi qua file tạm rồi os.replace: đổi tên là thao tác NGUYÊN TỬ trên cùng
        # một filesystem. Ghi thẳng thì người khác đọc đúng lúc đang ghi sẽ nhận file
        # cụt và JSON vỡ - hiếm nhưng có thật, và mất trắng chứ không báo lỗi gì.
        p = _ws_path(name)
        tmp = p + ".tmp"
        with open(tmp, "wb") as f:
            f.write(raw)
            f.flush()
            os.fsync(f.fileno())
        existed = os.path.exists(p)
        os.replace(tmp, p)
        return {"ok": True, "name": name, "size": len(raw), "overwrote": existed}

    @app.get("/video")
    def video(video_id: str):
        """Phát thẳng video gốc, không cắt không mã hoá lại.

        Trước đây muốn xem cả video phải mở dataset trên Kaggle - chậm và rời khỏi
        app. FileResponse của Starlette tự xử lý header Range, nên trình duyệt tua
        được bình thường và chỉ tải phần đang xem. Không tốn CPU vì không đụng ffmpeg.
        """
        src = video_path.get(video_id)
        if not src:
            raise HTTPException(404, f"không thấy video {video_id}")
        return FileResponse(src, media_type="video/mp4")

    @app.get("/keyframes")
    def keyframes(video_id: str, n_from: int = 1, n_to: int = 0):
        """Danh sách keyframe THẬT của một video: n, frame_id, pts_time.

        Cần endpoint riêng vì KHÔNG CÓ cách nào suy frame_id từ n ở phía web, mà
        khoảng cách giữa hai keyframe thì KHÔNG ĐỀU. Ví dụ L30_V078:
            n=31 -> 1848 ; n=32 -> 1893 ; n=33 -> 1917
        tức 45 rồi 24 frame, không phải fps=25. Suy bằng cách cộng dồn fps là sai
        ngay từ ô kế bên và sai tích luỹ càng xa càng nặng - ảnh thumbnail lấy theo
        n thì đúng, còn clip lấy theo frame_id suy ra thì trỏ sang cảnh khác.

        n_to = 0 nghĩa là lấy tới hết.
        """
        if video_id not in resolver._frames:
            raise HTTPException(404, f"không thấy video {video_id}")
        frames = resolver._frames[video_id]
        ns = resolver._n[video_id]
        pts = resolver._pts[video_id]
        out = []
        for i in range(len(frames)):
            n = int(ns[i])
            if n < n_from or (n_to and n > n_to):
                continue
            out.append({"n": n, "frame_id": int(frames[i]), "pts_time": float(pts[i])})
        return {"video_id": video_id, "fps": float(resolver._fps[video_id]),
                "count": len(out), "keyframes": out}

    @app.get("/clip")
    def clip(video_id: str, frame_id: int, seconds: float = 5.0):
        """Clip 5s quanh frame. MÃ HOÁ LẠI, không dùng -c copy.

        `-c copy` nhảy tới keyframe gần nhất nên clip không bắt đầu đúng giây yêu cầu —
        sai lệch có thể vài chục frame. Bộ đếm frame trên web dựa vào mốc bắt đầu để
        quy đổi thời gian -> frame, nên lệch mốc là lệch cả bộ đếm. Với TRAKE (khoảng
        đúng dưới 10 frame) thì hỏng hẳn. Mã hoá lại tốn ~1-2s nhưng chính xác tới frame.
        """
        src = video_path.get(video_id)
        if not src:
            raise HTTPException(404, f"không thấy video {video_id}")

        hit, fps, n, first, start = _clip_window(video_id, frame_id, seconds)
        out = f"/tmp/clip_{video_id}_{first}_{n}.mp4"

        if not os.path.exists(out):
            subprocess.run(
                ["ffmpeg", "-y", "-threads", "4", "-accurate_seek", "-ss", f"{start:.3f}", "-i", src,
                 "-t", str(seconds), "-c:v", "libx264", "-preset", "ultrafast",
                 "-crf", "28", "-c:a", "aac", "-b:a", "64k",
                 "-movflags", "+faststart",
                 "-vsync", "cfr", "-r", f"{fps:g}", out],
                check=True, capture_output=True,
            )
        return Response(
            open(out, "rb").read(), media_type="video/mp4",
            headers={
                "Cache-Control": "public, max-age=3600",
                "X-Clip-Start": f"{start:.3f}",
                "X-Clip-Fps": f"{fps:g}",
                "X-Clip-First-Frame": str(first),
                "Access-Control-Expose-Headers": "X-Clip-Start, X-Clip-Fps, X-Clip-First-Frame",
            },
        )

    return app


def _port_free(port: int) -> bool:
    import socket
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def serve(engine, analyze_fn=None, port: int = 8000,
          tunnel_token: str = None, hostname: str = None, access_key: str = None):
    """Chạy API trong thread nền + mở tunnel. Trả về (server, url)."""
    import uvicorn

    subprocess.run(["pkill", "-f", "cloudflared tunnel"], capture_output=True)
    time.sleep(0.5)

    if not _port_free(port):
        raise RuntimeError(
            f"Port {port} đang bị server cũ chiếm. Named tunnel ghim cứng port này trên "
            f"Cloudflare nên không nhảy port được.\n-> Restart kernel rồi chạy lại."
        )

    class NonBlocking(uvicorn.Server):
        def install_signal_handlers(self):
            pass

    access_key = access_key or make_key()
    # endpoint /image là `def` (đồng bộ) nên FastAPI chạy nó trong threadpool.
    # Mặc định 40 luồng -> 100 ảnh cùng lúc là xếp hàng. Nâng lên cho thoáng.
    try:
        import anyio.to_thread
        anyio.to_thread.current_default_thread_limiter().total_tokens = 160
    except Exception:
        pass

    app = build_app(engine, analyze_fn, access_key=access_key)
    # bind 0.0.0.0: trong container, `localhost` mà cloudflared dùng có thể là IPv6 ::1
    server = NonBlocking(uvicorn.Config(app, host="0.0.0.0", port=port, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(100):
        if server.started:
            break
        time.sleep(0.1)
    if not server.started:
        raise RuntimeError(f"Server không khởi động được ở port {port}.")
    print(f"✅ API chạy nền ở 0.0.0.0:{port}")

    if not os.path.exists("./cloudflared"):
        print("⏳ tải cloudflared...")
        subprocess.run(["wget", "-q", CLOUDFLARED_URL, "-O", "./cloudflared"], check=True)
        os.chmod("./cloudflared", 0o755)

    # ---- named tunnel: hostname cố định ----
    if tunnel_token:
        log = open("/tmp/cloudflared.log", "w")
        subprocess.Popen(["./cloudflared", "tunnel", "run", "--token", tunnel_token],
                         stdout=log, stderr=subprocess.STDOUT)
        # Chờ kết nối tunnel chủ động thay vì sleep cố định
        txt = ""
        for _ in range(30):
            time.sleep(0.2)
            try:
                txt = open("/tmp/cloudflared.log").read()
                if "Registered tunnel connection" in txt or "Connection registered" in txt:
                    break
            except Exception:
                pass
        url = f"https://{hostname}" if hostname else None
        print("\n" + "=" * 62)
        print(f"  URL CỐ ĐỊNH:  {url or '(xem Public Hostname trên Cloudflare)'}")
        print("=" * 62)
        if "Registered tunnel connection" in txt:
            print("\n✅ Tunnel đã kết nối.")
        else:
            print("\n❌ Chưa đăng ký được connector. 30 dòng log cuối:")
            print("\n".join(txt.splitlines()[-30:]) or "(rỗng)")

        print(f"\n  🔑 KEY:  {access_key}")
        print("     Dán vào web. Gửi cho đồng đội để cả đội dùng chung backend này.")
        print(f"\n502 -> Public Hostname phải là HTTP + localhost:{port}")
        return server, url

    # ---- quick tunnel: URL ngẫu nhiên ----
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    pat = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    url = None
    for _ in range(120):
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.25)
            continue
        m = pat.search(line)
        if m:
            url = m.group(0)
            break
    print("\n" + "=" * 62)
    print(f"  URL CÔNG KHAI:  {url or '❌ không lấy được'}")
    print("=" * 62)
    print(f"\n  🔑 KEY:  {access_key}")
    print("\n⚠️  URL đổi mỗi lần chạy - dán cả URL và key vào web.")


In [ ]:
# ===== CELL 8 - mở API ra internet =====
!pip install -q fastapi uvicorn

import sys
sys.path.insert(0, "/kaggle/working")
for m in ("api_server", "frame_resolver"):
    sys.modules.pop(m, None)
from api_server import serve

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("CF_TUNNEL_TOKEN")
except Exception:
    token = None
    print("ℹ️  Không có CF_TUNNEL_TOKEN -> dùng quick tunnel (URL đổi mỗi phiên).")

server, url = serve(
    engine,                      # engine tạo ở cell 5
    analyze_fn=analyze_prompt,   # để web gửi prompt thô, LLM tự tách + tự nhận Task
    port=8000,
    tunnel_token=token,
    hostname="aic.verse.id.vn",
)